In [ ]:
'''CASMI26 | Molecule Finder

Standalone retrieval baseline in the Kaggriculture style.

It turns a molecule's MS/MS spectra into an ordered list of chemically valid
SMILES: exact-mass candidates, direct library evidence, mass-shifted analog
evidence, and metric-exact de-duplication. It deliberately has no public-LB
candidate cap.

Attach the competition data. Optionally attach a public COCONUT CSV with a
canonical_smiles or smiles column to extend recall.

Output: submission.csv and diagnostics.json
'''


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import glob

WORK = Path('/kaggle/working' if Path('/kaggle/working').is_dir() else Path.cwd())
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, str(WORK))

try:
    import rdkit
except ImportError:
    wheels = glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True)
    if not wheels:
        raise RuntimeError('Attach an offline RDKit wheel; internet stays disabled for submission.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-index', wheels[0]])
    import rdkit
print('RDKit:', rdkit.__version__)

SETTINGS = {
    'ppm_window': 10.0,
    'analog_window_da': 200.0,
    'analog_count': 32,
    'peak_tolerance_da': 0.01,
    'candidate_csv_name': None,  # e.g. 'coconut.csv'; None = train structures only
    'top_n': 25,
}
(WORK / 'settings.json').write_text(json.dumps(SETTINGS, indent=2, sort_keys=True) + '\n')
print('Workdir:', WORK)
print(json.dumps(SETTINGS, indent=2))


In [ ]:
%%writefile chemistry.py
# SPDX-License-Identifier: Apache-2.0
"""Mass, structure identity, and submission rules."""
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.MolStandardize import rdMolStandardize

ELECTRON = 0.000548579909
PROTON = 1.007276466621
WATER = 18.01056468403
FORMIC_ACID = 46.0054793036
ADDUCTS = {
    '[M+H]+': (1, PROTON), '[M-H]-': (1, -PROTON),
    '[M+Na]+': (1, 22.989218), '[M+K]+': (1, 38.963158),
    '[M+NH4]+': (1, 18.033823), '[M+Cl]-': (1, 34.969402),
    '[M-H2O+H]+': (1, PROTON - WATER), '[M-2H2O+H]+': (1, PROTON - 2 * WATER),
    '[M+CH2O2-H]-': (1, FORMIC_ACID - PROTON),
    '[M+2H]2+': (2, 2 * PROTON), '[M-2H]2-': (2, -2 * PROTON),
}
_TAUTOMER = rdMolStandardize.TautomerEnumerator()

def neutral_mass(precursor_mz, adduct):
    try:
        charge, shift = ADDUCTS[str(adduct)]
        return float(precursor_mz) * charge - shift
    except (KeyError, TypeError, ValueError):
        return float('nan')

def molecule(smiles):
    return Chem.MolFromSmiles(str(smiles))

def structure_key(smiles):
    mol = molecule(smiles)
    if mol is None:
        return None
    canonical = _TAUTOMER.Canonicalize(mol)
    return Chem.MolToInchiKey(canonical).split('-')[0]

def exact_mass(smiles):
    mol = molecule(smiles)
    return None if mol is None else float(Descriptors.ExactMolWt(mol))

def deduplicate_top_smiles(smiles, limit=25):
    out, seen = [], set()
    for value in smiles:
        key = structure_key(value)
        if key and key not in seen:
            seen.add(key)
            out.append(value)
        if len(out) == limit:
            break
    return out

if __name__ == '__main__':
    assert abs(neutral_mass(101.007276466621, '[M+H]+') - 100.0) < 1e-8
    assert len(deduplicate_top_smiles(['O=C(O)C', 'CC(=O)O'])) == 1


In [ ]:
%%writefile finder.py
"""Small, inspectable evidence engine for CASMI26."""
from collections import defaultdict
from pathlib import Path
import glob
import json
import numpy as np
import pandas as pd
from rdkit import DataStructs
from rdkit.Chem import AllChem
from chemistry import deduplicate_top_smiles, exact_mass, neutral_mass

def find_file(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if not hits:
        raise FileNotFoundError(f'Missing {name}; attach the competition data.')
    return Path(sorted(hits, key=len)[0])

def cosine(a_mz, a_it, b_mz, b_it, tolerance):
    a_mz, a_it = np.asarray(a_mz), np.asarray(a_it, dtype=float)
    b_mz, b_it = np.asarray(b_mz), np.asarray(b_it, dtype=float)
    if not len(a_mz) or not len(b_mz):
        return 0.0
    a_it = a_it / max(a_it.max(), 1e-12); b_it = b_it / max(b_it.max(), 1e-12)
    i = j = 0; dot = 0.0
    while i < len(a_mz) and j < len(b_mz):
        delta = a_mz[i] - b_mz[j]
        if abs(delta) <= tolerance:
            dot += a_it[i] * b_it[j]; i += 1; j += 1
        elif delta < 0:
            i += 1
        else:
            j += 1
    return dot / max(np.linalg.norm(a_it) * np.linalg.norm(b_it), 1e-12)

def fp(smiles):
    from chemistry import molecule
    mol = molecule(smiles)
    return None if mol is None else AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)

class MoleculeFinder:
    def __init__(self, settings):
        self.s = settings

    def candidate_pool(self, train):
        smiles = list(dict.fromkeys(train.normalized_smiles.dropna().astype(str)))
        csv_name = self.s['candidate_csv_name']
        if csv_name:
            extra = pd.read_csv(find_file(csv_name))
            column = 'canonical_smiles' if 'canonical_smiles' in extra else 'smiles'
            smiles.extend(extra[column].dropna().astype(str))
            smiles = list(dict.fromkeys(smiles))
        rows = [(s, exact_mass(s), fp(s)) for s in smiles]
        rows = [(s, m, f) for s, m, f in rows if m is not None and f is not None]
        rows.sort(key=lambda row: row[1])
        if not rows:
            raise ValueError('Candidate pool is empty after RDKit validation.')
        self.pool = rows
        self.masses = np.array([row[1] for row in rows])

    def fit(self, train):
        train = train.copy()
        train['neutral_mass'] = [neutral_mass(m, a) for m, a in zip(train.precursor_mz, train.adduct)]
        train = train[np.isfinite(train.neutral_mass)]
        self.candidate_pool(train)
        self.library = train.sort_values('neutral_mass')
        self.library_mass = self.library.neutral_mass.to_numpy()

    def mass_slice(self, mass, ppm):
        delta = mass * ppm / 1e6
        return np.searchsorted(self.masses, [mass - delta, mass + delta])

    def score_molecule(self, spectra, mass):
        lo, hi = self.mass_slice(mass, self.s['ppm_window'])
        candidates = self.pool[lo:hi]
        if not candidates:
            lo, hi = self.mass_slice(mass, 30.0)
            candidates = self.pool[lo:hi]
        if not candidates:
            candidates = [self.pool[int(np.argmin(np.abs(self.masses - mass)))]]
        delta = mass * self.s['ppm_window'] / 1e6
        left, right = np.searchsorted(self.library_mass, [mass - delta, mass + delta])
        hits = self.library.iloc[left:right]
        direct = defaultdict(float)
        for row in hits.itertuples():
            direct[row.normalized_smiles] = max(direct[row.normalized_smiles], max(
                cosine(mz, it, row.ms2_mzs, row.ms2_normalized_intensities, self.s['peak_tolerance_da'])
                for mz, it in spectra))
        analog_rows = self.library.iloc[np.searchsorted(self.library_mass, max(0, mass-self.s['analog_window_da'])):
                                         np.searchsorted(self.library_mass, mass+self.s['analog_window_da'])]
        # ponytail: sample at most ~2k analog spectra; replace with indexed ANN search if this becomes the bottleneck.
        analog_rows = analog_rows.iloc[::max(1, len(analog_rows)//2048)]
        analogs = []
        for row in analog_rows.itertuples():
            score = max(cosine(mz, it, row.ms2_mzs, row.ms2_normalized_intensities, self.s['peak_tolerance_da']) for mz, it in spectra)
            if score > 0:
                analogs.append((score, fp(row.normalized_smiles)))
        analogs.sort(reverse=True, key=lambda item: item[0])
        analogs = analogs[:self.s['analog_count']]
        ranked = []
        for smiles, _, candidate_fp in candidates:
            analog = max((score**4 * DataStructs.TanimotoSimilarity(candidate_fp, afp) for score, afp in analogs), default=0.0)
            ranked.append((direct.get(smiles, 0.0) * 4.0 + analog, smiles))
        return [smiles for _, smiles in sorted(ranked, reverse=True)]

    def predict(self, test):
        rows, diagnostics = [], []
        for molecule_id, group in test.groupby('molecule_id', sort=False):
            masses = [neutral_mass(m, a) for m, a in zip(group.precursor_mz, group.adduct)]
            mass = float(np.nanmedian(masses))
            spectra = list(zip(group.ms2_mzs, group.ms2_normalized_intensities))
            ranked = self.score_molecule(spectra, mass) if np.isfinite(mass) else []
            top = deduplicate_top_smiles(ranked, self.s['top_n'])
            if not top:
                top = [self.pool[0][0]]  # valid format fallback; diagnostics keeps this visible.
            rows.append({'molecule_id': molecule_id, 'smiles': ';'.join(top)})
            diagnostics.append({'molecule_id': molecule_id, 'neutral_mass': mass, 'candidates': len(ranked), 'emitted': len(top)})
        return pd.DataFrame(rows), pd.DataFrame(diagnostics)

def run(settings):
    train = pd.read_parquet(find_file('train.parquet'))
    test = pd.read_parquet(find_file('test.parquet'))
    finder = MoleculeFinder(settings); finder.fit(train)
    predicted, diagnostics = finder.predict(test)
    template = pd.read_csv(find_file('sample_submission.csv'))[['molecule_id']]
    submission = template.merge(predicted, on='molecule_id', how='left', validate='one_to_one')
    if submission.smiles.isna().any():
        raise ValueError('Submission template contains molecule_ids absent from test predictions.')
    return submission, diagnostics


In [ ]:
from chemistry import deduplicate_top_smiles, neutral_mass

assert abs(neutral_mass(101.007276466621, '[M+H]+') - 100.0) < 1e-8
assert len(deduplicate_top_smiles(['CC(=O)O', 'O=C(O)C'])) == 1
print('Chemistry checks passed.')


In [ ]:
from finder import run

submission, diagnostics = run(SETTINGS)
assert list(submission.columns) == ['molecule_id', 'smiles']
assert submission.molecule_id.is_unique
assert submission.smiles.notna().all()
assert submission.smiles.str.len().gt(0).all(), 'Empty SMILES would be rejected by the scorer.'
assert (submission.smiles.str.count(';') < SETTINGS['top_n']).all()
submission.to_csv('submission.csv', index=False)
diagnostics.to_json('diagnostics.json', orient='records', indent=2)
print('Wrote submission.csv:', submission.shape)
display(diagnostics.describe(include='all'))
